# RAW to ACES Render Time Comparison

Measure and compare conversion times for converting RAW images to ACES format using rawtoaces.

## Import Required Libraries

Import necessary libraries for file handling, timing operations, and displaying results.

In [31]:
import os
import glob
import subprocess
import time
from pathlib import Path

import numpy as np

## Load Sample Images from Dataset

Locate the first 10 RAW image files from `dataset/temp/raw` directory.

In [32]:
# Define the dataset path using absolute path
notebooks_dir = Path.cwd()
raw_dir = (notebooks_dir.parent / "dataset" / "temp" / "raw").resolve()

# Get all RAW files (CR2, NEF, ARW, RAF, DNG)
raw_extensions = ["*.CR2", "*.NEF", "*.ARW", "*.RAF", "*.DNG"]
raw_files = []
for ext in raw_extensions:
    raw_files.extend(sorted(raw_dir.glob(ext)))

# Select first 10 files
sample_files = raw_files[:10]

print(f"Raw directory: {raw_dir}")
print(f"Total RAW files found: {len(raw_files)}")
print(f"Selected {len(sample_files)} files for conversion:")
for i, f in enumerate(sample_files, 1):
    print(f"  {i}. {f.name}")

Raw directory: /run/media/mikkelkp/Projects/Projects/med8_project/LuminaScale/dataset/temp/raw
Total RAW files found: 1000
Selected 10 files for conversion:
  1. 0_0.CR2
  2. 0_1.CR2
  3. 0_2.CR2
  4. 0_3.CR2
  5. 0_4.CR2
  6. 0_5.CR2
  7. 0_6.CR2
  8. 1000_0.CR2
  9. 1000_1.CR2
  10. 1000_2.CR2


## Convert RAW Images to ACES Format

Convert each RAW image to ACES format using the rawtoaces command.

In [33]:
# Setup output directory using absolute path
notebook_dir = Path.cwd()
output_dir = notebook_dir.parent / "dataset" / "temp" / "aces_converted"
output_dir.mkdir(parents=True, exist_ok=True)

# Also create with absolute path for rawtoaces
output_dir_abs = output_dir.resolve()

# Storage for timing data
conversion_times = []
results = []

print(f"Notebook directory: {notebook_dir}")
print(f"Output directory: {output_dir_abs}\n")
print("Starting conversions...")
print("-" * 60)

Notebook directory: /run/media/mikkelkp/Projects/Projects/med8_project/LuminaScale/notebooks
Output directory: /run/media/mikkelkp/Projects/Projects/med8_project/LuminaScale/dataset/temp/aces_converted

Starting conversions...
------------------------------------------------------------


In [34]:
# Convert each file and record timing, then display statistics
for idx, raw_file in enumerate(sample_files, 1):
    filename = raw_file.name
    output_path = output_dir_abs / f"{raw_file.stem}.exr"
    
    # Measure conversion time
    start_time = time.time()
    
    try:
        # Run rawtoaces command with proper flags (matching bash script)
        cmd = [
            "rawtoaces",
            "--data-dir", "/usr/local/share/rawtoaces/data",
            "--output-dir", str(output_dir_abs),
            "--create-dirs",
            "--overwrite",
            str(raw_file.resolve())
        ]
        result = subprocess.run(cmd, capture_output=True, text=True, timeout=300)
        
        elapsed_time = time.time() - start_time
        
        if result.returncode == 0:
            status = "Success"
        else:
            status = "Failed"
        
        conversion_times.append(elapsed_time)
        results.append({
            "Image": filename,
            "Time (seconds)": elapsed_time,
            "Status": status,
            "Output": output_path.name
        })
        
    except subprocess.TimeoutExpired:
        conversion_times.append(None)
        results.append({
            "Image": filename,
            "Time (seconds)": "Timeout",
            "Status": "Timeout",
            "Output": None
        })
    except Exception as e:
        conversion_times.append(None)
        results.append({
            "Image": filename,
            "Time (seconds)": "Error",
            "Status": "Error",
            "Output": None
        })

# Calculate and display statistics for successful conversions
valid_times = [t for t in conversion_times if isinstance(t, (int, float))]

if valid_times:
    total_time = sum(valid_times)
    avg_time = total_time / len(valid_times)
    min_time = min(valid_times)
    max_time = max(valid_times)
    
    print(f"\nConversion Time Statistics:")
    print(f"  Total time: {total_time:.2f} seconds")
    print(f"  Average time per image: {avg_time:.2f} seconds")
    print(f"  Minimum time: {min_time:.2f} seconds")
    print(f"  Maximum time: {max_time:.2f} seconds")
    print(f"  Successfully converted: {len(valid_times)}/{len(sample_files)}")
else:
    print("\nNo successful conversions to report.")


Conversion Time Statistics:
  Total time: 10.93 seconds
  Average time per image: 1.09 seconds
  Minimum time: 0.97 seconds
  Maximum time: 1.18 seconds
  Successfully converted: 10/10


## ACES2065-1 to sRGB Conversion with OpenImageIO

Convert ACES EXR images to 8-bit sRGB format using OpenImageIO's color space conversion.


In [35]:
import numpy as np
import torch
import OpenImageIO as oiio
import imageio

# GPU-accelerated ACES to sRGB conversion using PyTorch
device = "cuda" if torch.cuda.is_available() else "cpu"

# ACES2065-1 to sRGB conversion matrices (using standard cinema pipeline)
ACES_TO_XYZ = torch.tensor([
    [0.6380, 0.2646, 0.0000],
    [0.2126, 0.7874, 0.0000],
    [0.0000, 0.0000, 1.0888]
], dtype=torch.float32, device=device)

XYZ_TO_SRGB = torch.tensor([
    [3.2406, -1.5372, -0.4986],
    [-0.9689, 1.8758, 0.0415],
    [0.0557, -0.2040, 1.0570]
], dtype=torch.float32, device=device)

def load_aces_and_convert_srgb_gpu(aces_path: Path | str) -> np.ndarray:
    """Load ACES2065-1 EXR and convert to sRGB using GPU-accelerated PyTorch."""
    # Load EXR using OIIO
    inp = oiio.ImageInput.open(str(aces_path))
    if not inp:
        raise RuntimeError(f"Failed to open {aces_path}")
    
    image = inp.read_image()
    inp.close()
    
    # Handle metadata if available
    h, w = inp.spec().height, inp.spec().width
    c = inp.spec().nchannels
    
    # Convert to torch tensor on GPU
    img_tensor = torch.from_numpy(image).to(device).float()
    img_tensor = img_tensor.reshape(h, w, c)
    
    # Extract RGB channels (drop alpha if present)
    if c >= 3:
        rgb = img_tensor[:, :, :3]
    else:
        rgb = img_tensor
    
    # Reshape for matrix multiplication: [H*W, 3]
    h_t, w_t = rgb.shape[:2]
    rgb_flat = rgb.reshape(-1, 3)
    
    # Apply color conversion: ACES2065-1 -> XYZ -> sRGB
    xyz_flat = torch.matmul(rgb_flat, ACES_TO_XYZ.T)
    srgb_flat = torch.matmul(xyz_flat, XYZ_TO_SRGB.T)
    
    # Reshape back to image
    srgb = srgb_flat.reshape(h_t, w_t, 3)
    
    # Apply sRGB gamma correction
    def srgb_gamma(x):
        return torch.where(
            x <= 0.0031308,
            12.92 * x,
            1.055 * torch.pow(x, 1.0 / 2.4) - 0.055
        )
    
    srgb = srgb_gamma(srgb)
    
    # Clamp and convert back to numpy
    srgb = torch.clamp(srgb, 0.0, 1.0)
    srgb_np = srgb.cpu().numpy().astype(np.float32)
    
    return srgb_np

# Setup directories
aces_src_dir = (Path.cwd().parent / "dataset" / "temp" / "aces").resolve()
aces_output_dir = (Path.cwd().parent / "dataset" / "temp" / "aces_converted").resolve()
aces_output_dir.mkdir(parents=True, exist_ok=True)

# Get all ACES EXR files
aces_files_list = sorted(list(aces_src_dir.glob("*_aces.exr")))

print(f"ACES source directory: {aces_src_dir}")
print(f"ACES output directory: {aces_output_dir}")
print(f"GPU device: {device}")
print(f"Total ACES images found: {len(aces_files_list)}")
print("-" * 60)

# Batch convert and measure timing
aces_conversion_times = []
aces_results = []

for idx, aces_path in enumerate(aces_files_list, 1):
    filename = aces_path.name
    output_path = aces_output_dir / f"{aces_path.stem}.png"
    
    start_time = time.time()
    
    try:
        # GPU-accelerated conversion
        srgb_image = load_aces_and_convert_srgb_gpu(aces_path)
        
        # Convert to 8-bit and save
        srgb_8bit = (np.clip(srgb_image, 0, 1) * 255).astype(np.uint8)
        imageio.imwrite(output_path, srgb_8bit)
        
        elapsed_time = time.time() - start_time
        status = "Success"
        aces_conversion_times.append(elapsed_time)
        aces_results.append({
            "Image": filename,
            "Time (seconds)": elapsed_time,
            "Status": status
        })
        
    except Exception as e:
        elapsed_time = time.time() - start_time
        print(f"Error processing {filename}: {e}")
        aces_conversion_times.append(None)
        aces_results.append({
            "Image": filename,
            "Time (seconds)": "Error",
            "Status": "Failed"
        })

print("-" * 60)

# Display statistics
aces_valid_times = [t for t in aces_conversion_times if isinstance(t, (int, float))]

if aces_valid_times:
    aces_total_time = sum(aces_valid_times)
    aces_avg_time = aces_total_time / len(aces_valid_times)
    aces_min_time = min(aces_valid_times)
    aces_max_time = max(aces_valid_times)
    
    print(f"\nACES to sRGB Conversion Time Statistics (GPU-accelerated with PyTorch):")
    print(f"  Total time: {aces_total_time:.2f} seconds")
    print(f"  Average time per image: {aces_avg_time:.2f} seconds")
    print(f"  Minimum time: {aces_min_time:.2f} seconds")
    print(f"  Maximum time: {aces_max_time:.2f} seconds")
    print(f"  Successfully converted: {len(aces_valid_times)}/{len(aces_files_list)}")
else:
    print("\nNo successful ACES conversions to report.")

ACES source directory: /run/media/mikkelkp/Projects/Projects/med8_project/LuminaScale/dataset/temp/aces
ACES output directory: /run/media/mikkelkp/Projects/Projects/med8_project/LuminaScale/dataset/temp/aces_converted
GPU device: cpu
Total ACES images found: 10
------------------------------------------------------------
------------------------------------------------------------

ACES to sRGB Conversion Time Statistics (GPU-accelerated with PyTorch):
  Total time: 14.69 seconds
  Average time per image: 1.47 seconds
  Minimum time: 1.33 seconds
  Maximum time: 1.61 seconds
  Successfully converted: 10/10
